# Sales Intelligence Platform Project

## Import relevant libraries

In [ ]:
# installing required sql magic packages
!pip install sqlalchemy
!pip install ipython-sql
!pip install pymysql

In [91]:
import numpy as np
import pandas as pd

## Loading the data

In [92]:
FILE_PATH = 'data/raw_data/Sales Dataset - Sales Dataset.csv'

data = pd.read_csv(FILE_PATH)
df = data.copy() #making a copy to preserve the integrity of the original dataset
print(f'Data shape: {df.shape}\n')
df.head()

Data shape: (1194, 12)



,Order ID,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode,Order Date,CustomerName,State,City,Year-Month
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,Miami,2023-06
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2024-12
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,Buffalo,2021-07
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,Miami,2023-06
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2024-12


`Key observations:`
- `Order ID` is not unique. It appears the same for different customers, having different orders, state, city and date.
- `Year-Month` is not useful in the OLTP (repetition). It can be derived from `Order Date` when building the OLAP later.

## Data quality checks and Data cleaning

### Standardising the data columns

In [93]:
df.columns

Index(['Order ID', 'Amount', 'Profit', 'Quantity', 'Category', 'Sub-Category',
       'PaymentMode', 'Order Date', 'CustomerName', 'State', 'City',
       'Year-Month'],
      dtype='object')

In [94]:
import re

df.columns = (
    df.columns
    .str.strip()                                               #remove trailing and leading white spaces
    .map(lambda x: re.sub(r'(?<=[a-z])(?=[A-Z])', '_', x))     #replace camel case in headers with underscore
    .str.replace(r'[\s-]', '_', regex=True)                    #replace white spaces and hyphens in headers with underscore
    .str.lower()                                               #convert to lower case                                             
)

df.columns

Index(['order_id', 'amount', 'profit', 'quantity', 'category', 'sub_category',
       'payment_mode', 'order_date', 'customer_name', 'state', 'city',
       'year_month'],
      dtype='object')

### Dropping the year_month column

In [95]:
df.drop(columns=['year_month'], inplace=True)
df.head()

,order_id,amount,profit,quantity,category,sub_category,payment_mode,order_date,customer_name,state,city
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,Miami
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,Chicago
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,Buffalo
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,Miami
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,Chicago


### Duplicated values

In [96]:
# checking for duplicated values
df.duplicated().sum()

0

### Null values

In [97]:
# checking for null values
df.isnull().sum()

order_id         0
amount           0
profit           0
quantity         0
category         0
sub_category     0
payment_mode     0
order_date       0
customer_name    0
state            0
city             0
dtype: int64

### Investigating the Order ID issue earlier observed

In [98]:
total_orders = df['order_id'].nunique() #unique items in the order_id column
problematic_orders = df.groupby('order_id')['customer_name'].nunique()[lambda x: x > 1].count() #customers sharing the same order_id 

print(f'Total unique order IDs: {total_orders}')
print(f'Order IDs shared across multiple customers: {problematic_orders}')
print(f'Percentage problematic: {problematic_orders / total_orders * 100:.2f}%\n') #percentage occurrence in the dataset to measure severity

Total unique order IDs: 547
Order IDs shared across multiple customers: 194
Percentage problematic: 35.47%



It has returned that over a third of order_ids are shared across completely different customers. Therefore, it is unreliable as a unique identifier. This would be considered when designing the ERDs for the OLTP and OLAP. 

### General information on the dataset

In [99]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   order_id       1194 non-null   object
 1   amount         1194 non-null   int64 
 2   profit         1194 non-null   int64 
 3   quantity       1194 non-null   int64 
 4   category       1194 non-null   object
 5   sub_category   1194 non-null   object
 6   payment_mode   1194 non-null   object
 7   order_date     1194 non-null   object
 8   customer_name  1194 non-null   object
 9   state          1194 non-null   object
 10  city           1194 non-null   object
dtypes: int64(3), object(8)
memory usage: 102.7+ KB


### Standardising data types for amount, profit and order_date

In [100]:
df = df.astype({'amount': float, 'profit': float})
df['order_date'] = pd.to_datetime(df['order_date'])

In [101]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       1194 non-null   object        
 1   amount         1194 non-null   float64       
 2   profit         1194 non-null   float64       
 3   quantity       1194 non-null   int64         
 4   category       1194 non-null   object        
 5   sub_category   1194 non-null   object        
 6   payment_mode   1194 non-null   object        
 7   order_date     1194 non-null   datetime64[ns]
 8   customer_name  1194 non-null   object        
 9   state          1194 non-null   object        
 10  city           1194 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(7)
memory usage: 102.7+ KB


## Defining the OLTP tables

In [ ]:
# customers
customers = df[['customer_name']].copy().drop_duplicates().reset_index(drop=True)

In [102]:
df.columns

Index(['order_id', 'amount', 'profit', 'quantity', 'category', 'sub_category',
       'payment_mode', 'order_date', 'customer_name', 'state', 'city'],
      dtype='object')

In [ ]:
# locations
locations = df[['city', 'state']].copy().drop_duplicates().reset_index(drop=True)

In [ ]:
# payment_methods
payment_methods = df[['payment_mode']].copy().drop_duplicates().reset_index(drop=True)

In [ ]:
# products
products = df[['category', 'sub_category']].copy().drop_duplicates().reset_index(drop=True)

In [ ]:
# orders
products = df[['order_id', 'customer_id', 'location_id', 'order_id']].copy().drop_duplicates().reset_index(drop=True)

In [ ]:
# order items